# 02 · What drives global trait space? XGBoost + SHAP with environmentally and spatially separated cross-validation

This notebook follows Mahecha et al. (2021, *Ecography*), with two changes: XGBoost replaces random forest, and SHAP values replace random-forest variable importance.

**Targets:** the trait PCs per H3 res-3 cell, in three modes:

- **present**: all species
- **native**: native species only
- **shift**: present − native. Both time steps share one PCA rotation, so the difference is meaningful.

**Predictors:**

- PCA of climate (4 PCs)
- PCA of soil (4 PCs)
- log population density, as the sampling-bias covariate, as in Mahecha et al.
- land-cover fractions

**Cross-validation:** affinity propagation on an asymmetric similarity matrix. The lower triangle holds similarity in trait space and the upper triangle holds geographic proximity, which gives folds that are both spatially contiguous and environmentally distinct. Each fold is left out in turn.

**Inputs** (same folder as this notebook):

- your trait CSV (long format: `h3_res3`, `group` ∈ {all, native}, `PC1…PC5`, `count`, …)
- `env_data/env_predictors_h3r3.csv` from notebook 01

**Outputs:** everything is written to `results_trait_env/`, with figures in `results_trait_env/figures/`.

In [3]:
# %pip install numpy pandas scipy scikit-learn xgboost matplotlib pyarrow   # once, if missing (cartopy optional, for coastlines)
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr
from sklearn.cluster import AffinityPropagation
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False

print("xgboost", xgb.__version__, "| cartopy:", HAS_CARTOPY)

xgboost 2.1.4 | cartopy: True


## Settings
Everything you might want to change lives in this one cell.

In [10]:
# --- Inputs / outputs -----------------------------------------------------------
TRAIT_CSV = Path("/net/scratch/swolf/GBIF/gbif_res3_try6_CS_wcvp_summary_stats_inkl_pca_mean_group_region_log10_notrim_misstree.csv")
ENV_DIR   = Path("~/Func_Neo_Phytes/Data/env_data").expanduser()
ENV_PATTERN = "env_predictors_h3r3*.csv"
OUT_DIR   = Path("~/Func_Neo_Phytes/results_trait_env").expanduser()   # or wherever you want the results
FIG_DIR   = OUT_DIR / "figures"

# --- What to model --------------------------------------------------------------
MODES     = ["present", "native", "shift"]
TRAIT_PCS = ["PC1", "PC2", "PC3"]          # targets (PC1 size, PC2 leaf economics, PC3 turnover)

# --- Predictors -----------------------------------------------------------------
N_ENV_PCS    = 4        # PCs kept per block (climate, soil), as in Mahecha et al.
USE_POP      = True     # log10(pop density + 1) as sampling-bias covariate
USE_LANDCOVER = True    # land-cover fractions
POP_SOURCE = "GPW v4 2020"   # for the methods text
LANDCOVER_SOURCE = "per-cell land-cover fractions (urban, cropland, natural vegetation, barren, water)"  # for the methods text: add product + year
MAX_BLOCK_NA = 0.5      # drop a cell if more than this share of its climate or soil variables is missing

# --- Spatial folds (affinity propagation) ----------------------------------------
AP_TRAIT_PCS   = ["PC1", "PC2", "PC3", "PC4"]  # "environmental" side of the similarity = present-day trait space
AP_SUBSAMPLE   = 4000   # AP needs several n×n matrices; on a subsample of 4,000 that is ~0.5 GB. None = all cells
AP_PREFERENCE_QUANTILE = 0.0   # 0 = min similarity → fewest clusters; raise (e.g. 0.05) for more folds
AP_DAMPING     = 0.9
MIN_FOLD_SIZE  = 50     # smaller clusters are merged into their most similar neighbour

# --- XGBoost --------------------------------------------------------------------
XGB_PARAMS = dict(
    n_estimators=3000, learning_rate=0.03, max_depth=6, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist", n_jobs=-1, random_state=42,
)
EARLY_STOPPING = 100
SEED = 42

FIG_DIR.mkdir(parents=True, exist_ok=True)

In [6]:
# Figure style: light surface, recessive axes; blue/orange/aqua for the three modes,
# one-hue blue for magnitudes, blue↔red around a grey midpoint for signed values.
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
MODE_COLORS = {"present": "#2a78d6", "native": "#eb6834", "shift": "#1baf7a"}
SEQ = LinearSegmentedColormap.from_list(
    "seq_blue", ["#f3f7fd", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
DIV = LinearSegmentedColormap.from_list(
    "div_blue_red", ["#104281", "#3987e5", "#9ec5f4", "#f0efec", "#f3aaa9", "#e34948", "#9e2626"])

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
    "xtick.color": INK2, "ytick.color": INK2, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "legend.frameon": False, "savefig.dpi": 200, "savefig.bbox": "tight",
})

def savefig(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png")
    fig.savefig(FIG_DIR / f"{name}.pdf")

## 1 · Trait targets
The trait file has one row per cell and group. It is split into present (`group == "all"`) and native, and the shift is computed as present − native.

In [7]:
traits_long = pd.read_csv(TRAIT_CSV).rename(columns={"h3_res3": "h3_index"})
pc_cols = sorted([c for c in traits_long.columns if re.fullmatch(r"PC\d+", c)], key=lambda c: int(c[2:]))

present = traits_long[traits_long["group"] == "all"].set_index("h3_index")
native  = traits_long[traits_long["group"] == "native"].set_index("h3_index")
trait_cells = present.index.intersection(native.index)

TARGETS = {
    "present": present.loc[trait_cells, pc_cols],
    "native":  native.loc[trait_cells, pc_cols],
    "shift":   present.loc[trait_cells, pc_cols] - native.loc[trait_cells, pc_cols],
}
cell_info = pd.DataFrame({"n_records_all": present.loc[trait_cells, "count"],
                          "n_records_native": native.loc[trait_cells, "count"]})

print(f"{len(trait_cells)} cells with both present and native values; trait PCs available: {pc_cols}")
zero_shift = (TARGETS["shift"][TRAIT_PCS].abs().sum(axis=1) == 0).mean()
print(f"{zero_shift:.1%} of cells have exactly zero shift (no introduced species recorded)")
pd.concat({m: TARGETS[m][TRAIT_PCS].describe().T[["mean", "std", "min", "max"]] for m in MODES}).round(3)

9752 cells with both present and native values; trait PCs available: ['PC1', 'PC2', 'PC3', 'PC4', 'PC5']
18.9% of cells have exactly zero shift (no introduced species recorded)


mean    std     min     max
present PC1 -0.025  3.592 -13.106  16.905
        PC2  0.128  2.482 -13.190  12.831
        PC3 -0.099  1.681 -11.272   8.890
native  PC1  0.025  3.749 -13.096  16.949
        PC2 -0.128  2.606 -13.196  12.819
        PC3  0.099  1.771 -11.245   8.939
shift   PC1 -0.050  0.900 -11.924  14.307
        PC2  0.255  0.684 -10.623   8.467
        PC3 -0.199  0.490  -8.418   5.272

## 2 · Environmental predictors
The notebook-01 CSVs are read in, and climate and soil are each reduced to `N_ENV_PCS` principal components on standardised variables. Population density is log-transformed. Land-cover fractions go in as they are: there are only a few, they are bounded in [0, 1], and they are directly interpretable.

In [11]:
env_files = sorted(ENV_DIR.glob(ENV_PATTERN))
assert env_files, f"No files matching {ENV_DIR}/{ENV_PATTERN}. Run notebook 01 first."
env = (pd.concat([pd.read_csv(f) for f in env_files])
         .drop_duplicates("h3_index").set_index("h3_index"))

wc_cols = [c for c in env.columns if c.startswith("wc_")]
sg_cols = [c for c in env.columns if c.startswith("sg_")]
lu_cols = [c for c in env.columns if c.startswith("lu_frac_")]
print(f"{len(env_files)} file(s), {len(env)} cells | climate {len(wc_cols)} vars, soil {len(sg_cols)} vars, "
      f"land cover {len(lu_cols)} classes, population: {'pop_density' in env.columns}")

na_wc, na_sg = env[wc_cols].isna().mean(axis=1), env[sg_cols].isna().mean(axis=1)
keep = (na_wc <= MAX_BLOCK_NA) & (na_sg <= MAX_BLOCK_NA)
print(f"dropping {(~keep).sum()} cells with >{MAX_BLOCK_NA:.0%} missing climate or soil values "
      f"(typically ice, tiny islands, coast-only cells)")
env = env[keep]

1 file(s), 9752 cells | climate 103 vars, soil 78 vars, land cover 5 classes, population: True
dropping 2819 cells with >50% missing climate or soil values (typically ice, tiny islands, coast-only cells)


In [12]:
def block_pca(df, cols, n, label):
    """Standardise, impute medians, PCA. Loadings are correlations between variables and PCs."""
    n = min(n, len(cols))                      # in case a block has very few variables
    X = df[cols].astype(float)
    X = X.fillna(X.median())
    Z = StandardScaler().fit_transform(X)
    pca = PCA(n_components=n, random_state=SEED).fit(Z)
    names = [f"{label}_PC{i+1}" for i in range(n)]
    scores = pd.DataFrame(pca.transform(Z), index=df.index, columns=names)
    loadings = pd.DataFrame(pca.components_.T * np.sqrt(pca.explained_variance_), index=cols, columns=names)
    return scores, loadings, pca.explained_variance_ratio_

clim_scores, clim_load, clim_var = block_pca(env, wc_cols, N_ENV_PCS, "clim")
soil_scores, soil_load, soil_var = block_pca(env, sg_cols, N_ENV_PCS, "soil")

for name, var in [("climate", clim_var), ("soil", soil_var)]:
    print(f"{name:8s} PCs explain " + ", ".join(f"{v:.0%}" for v in var) + f"  (total {var.sum():.0%})")

def top_loadings(load, k=4):
    """For each PC: the k variables with the strongest correlation, with sign."""
    return pd.DataFrame({pc: [f"{v} ({load.loc[v, pc]:+.2f})" for v in load[pc].abs().nlargest(k).index]
                         for pc in load.columns})

display(top_loadings(clim_load))
display(top_loadings(soil_load))

climate  PCs explain 53%, 15%, 10%, 7%  (total 85%)
soil     PCs explain 33%, 23%, 14%, 8%  (total 77%)


,clim_PC1,clim_PC2,clim_PC3,clim_PC4
0,wc_tmin_04 (-0.97),wc_bio12 (+0.68),wc_tmin_07 (-0.66),wc_wind_02 (+0.67)
1,wc_tmin_10 (-0.97),wc_srad_08 (-0.67),wc_tavg_07 (-0.64),wc_wind_01 (+0.67)
2,wc_bio01 (-0.97),wc_srad_04 (-0.67),wc_srad_05 (-0.59),wc_wind_12 (+0.63)
3,wc_bio11 (-0.97),wc_bio18 (+0.63),wc_srad_06 (-0.58),wc_wind_03 (+0.62)


,soil_PC1,soil_PC2,soil_PC3,soil_PC4
0,sg_wv0010_15-30cm_mean (+0.89),sg_clay_15-30cm_mean (-0.84),sg_phh2o_100-200cm_mean (-0.75),sg_bdod_100-200cm_mean (+0.51)
1,sg_wv0033_5-15cm_mean (+0.88),sg_clay_30-60cm_mean (-0.84),sg_phh2o_60-100cm_mean (-0.74),sg_silt_0-5cm_mean (+0.50)
2,sg_ocd_5-15cm_mean (+0.86),sg_wv1500_60-100cm_mean (-0.84),sg_phh2o_30-60cm_mean (-0.73),sg_silt_15-30cm_mean (+0.50)
3,sg_wv0033_15-30cm_mean (+0.85),sg_wv1500_30-60cm_mean (-0.83),sg_phh2o_15-30cm_mean (-0.71),sg_silt_5-15cm_mean (+0.50)


Use the two tables above to name the climate and soil axes in the paper, e.g. "clim_PC1 = temperature / seasonality". The PC signs are arbitrary.

In [13]:
FEATURES = pd.concat([clim_scores, soil_scores], axis=1)
if USE_POP:
    FEATURES["log_pop_density"] = np.log10(env.loc[FEATURES.index, "pop_density"].clip(lower=0) + 1)
    pop_missing = FEATURES["log_pop_density"].isna()
    print(f"dropping {pop_missing.sum()} cells without a population value")
    FEATURES = FEATURES[~pop_missing]
if USE_LANDCOVER:
    FEATURES = FEATURES.join(env.loc[FEATURES.index, lu_cols])
    lu_missing = FEATURES[lu_cols].isna().any(axis=1)
    print(f"dropping {lu_missing.sum()} cells without land-cover values (not in the table, or almost all water)")
    FEATURES = FEATURES[~lu_missing]

# keep only cells that have both traits and predictors
CELLS = FEATURES.index.intersection(trait_cells)
FEATURES = FEATURES.loc[CELLS]
LAT, LON = env.loc[CELLS, "lat"].to_numpy(), env.loc[CELLS, "lon"].to_numpy()

PRETTY = {"log_pop_density": "log pop. density"}
PRETTY.update({c: f"climate PC{c[-1]}" for c in clim_scores.columns})
PRETTY.update({c: f"soil PC{c[-1]}" for c in soil_scores.columns})
LU_NAMES = {"crop": "cropland", "nat": "natural veg.", "water": "water (share of cell)"}
PRETTY.update({c: LU_NAMES.get(c.replace("lu_frac_", ""), c.replace("lu_frac_", "")) + " cover" for c in lu_cols})
pretty = lambda c: PRETTY.get(c, c)

print(f"modelling {len(CELLS)} cells × {FEATURES.shape[1]} predictors: {list(FEATURES.columns)}")

dropping 0 cells without a population value
dropping 59 cells without land-cover values (not in the table, or almost all water)
modelling 6874 cells × 14 predictors: ['clim_PC1', 'clim_PC2', 'clim_PC3', 'clim_PC4', 'soil_PC1', 'soil_PC2', 'soil_PC3', 'soil_PC4', 'log_pop_density', 'lu_frac_urban', 'lu_frac_crop', 'lu_frac_nat', 'lu_frac_barren', 'lu_frac_water']


## 3 · Spatial folds: affinity propagation on an asymmetric similarity matrix
As in Mahecha et al., the lower triangle of the similarity matrix is the negative Euclidean distance in the target space. Here that is the leading present-day trait PCs (Mahecha et al. used Isomap dimensions). The upper triangle is the negative great-circle distance. Both are divided by their median, so neither dominates.

Affinity propagation picks the number of clusters itself. It is run on a random subsample (`AP_SUBSAMPLE`) to keep memory manageable, and the remaining cells join the most similar exemplar. Clusters smaller than `MIN_FOLD_SIZE` are merged into their closest neighbour.

In [14]:
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    dlat = lat2[None, :] - lat1[:, None]
    dlon = lon2[None, :] - lon1[:, None]
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1)[:, None] * np.cos(lat2)[None, :] * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))

F_ap = TARGETS["present"].loc[CELLS, AP_TRAIT_PCS].to_numpy()
n = len(CELLS)
rng = np.random.default_rng(SEED)
sub = np.sort(rng.choice(n, AP_SUBSAMPLE, replace=False)) if AP_SUBSAMPLE and n > AP_SUBSAMPLE else np.arange(n)

D_env = cdist(F_ap[sub], F_ap[sub])
D_geo = haversine_km(LAT[sub], LON[sub], LAT[sub], LON[sub])
iu = np.triu_indices(len(sub), 1)
s_env, s_geo = np.median(D_env[iu]), np.median(D_geo[iu])

lower = np.tril(np.ones_like(D_env, dtype=bool), -1)
S = np.where(lower, -D_env / s_env, -D_geo / s_geo)        # asymmetric similarity
preference = np.quantile(S[~np.eye(len(sub), dtype=bool)], AP_PREFERENCE_QUANTILE)
del D_env, D_geo

with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    ap = AffinityPropagation(affinity="precomputed", preference=preference, damping=AP_DAMPING,
                             max_iter=3000, convergence_iter=100, random_state=SEED).fit(S)
    if any("did not converge" in str(x.message) for x in w):
        print("⚠ affinity propagation did not converge. Try AP_DAMPING = 0.95.")
del S
exemplars = sub[ap.cluster_centers_indices_]
print(f"affinity propagation found {len(exemplars)} clusters on {len(sub)} cells")

affinity propagation found 39 clusters on 4000 cells


In [15]:
def combined_similarity(i_idx, j_idx):
    """Symmetric version (mean of both triangles) used to attach the remaining cells and merge small clusters."""
    de = cdist(F_ap[i_idx], F_ap[j_idx]) / s_env
    dg = haversine_km(LAT[i_idx], LON[i_idx], LAT[j_idx], LON[j_idx]) / s_geo
    return -(de + dg) / 2

folds = np.argmax(combined_similarity(np.arange(n), exemplars), axis=1)
folds[sub] = ap.labels_                       # subsampled cells keep their AP label

# merge clusters that are too small to be a meaningful test set
while True:
    sizes = np.bincount(folds, minlength=len(exemplars))
    alive = np.flatnonzero(sizes)
    small = [k for k in alive if sizes[k] < MIN_FOLD_SIZE]
    if not small or len(alive) <= 2:
        break
    k = min(small, key=lambda c: sizes[c])
    others = [c for c in alive if c != k]
    target = others[int(np.argmax(combined_similarity([exemplars[k]], exemplars[others])[0]))]
    folds[folds == k] = target

_, FOLDS = np.unique(folds, return_inverse=True)  # relabel 0..K-1
N_FOLDS = FOLDS.max() + 1
sizes = np.bincount(FOLDS)
print(f"{N_FOLDS} folds after merging; cells per fold: min {sizes.min()}, median {int(np.median(sizes))}, max {sizes.max()}")
pd.DataFrame({"h3_index": CELLS, "fold": FOLDS, "lat": LAT, "lon": LON}).to_csv(OUT_DIR / "spatial_folds.csv", index=False)

39 folds after merging; cells per fold: min 87, median 181, max 291


In [16]:
def map_axes(fig, pos):
    if HAS_CARTOPY:
        ax = fig.add_subplot(*pos, projection=ccrs.Robinson())
        ax.set_global()
        ax.add_feature(cfeature.COASTLINE, linewidth=0.3, edgecolor=INK2)
        return ax, dict(transform=ccrs.PlateCarree())
    ax = fig.add_subplot(*pos)
    ax.set_xlim(-180, 180); ax.set_ylim(-60, 85); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax, {}

fig = plt.figure(figsize=(11, 5.2))
ax, tf = map_axes(fig, (1, 1, 1))
# Many folds → colour only separates neighbours; every fold is also labelled with its number.
fold_colors = plt.get_cmap("tab20")(np.arange(N_FOLDS) % 20)
ax.scatter(LON, LAT, c=fold_colors[FOLDS], s=2.5, linewidths=0, **tf)
for k in range(N_FOLDS):
    m = FOLDS == k
    ax.text(np.median(LON[m]), np.median(LAT[m]), str(k + 1), fontsize=7, ha="center", va="center",
            color=INK, bbox=dict(boxstyle="round,pad=0.15", fc=SURFACE, ec="none", alpha=0.8), **tf)
ax.set_title(f"Spatial cross-validation folds ({N_FOLDS}, affinity propagation)", loc="left")
savefig(fig, "00_spatial_folds")
plt.show()

AttributeError: 'GeoAxes' object has no attribute '_autoscaleXon'

Error in callback <function _draw_all_if_interactive at 0x711ba1f70790> (for post_execute):


AttributeError: 'GeoAxes' object has no attribute '_autoscaleXon'

AttributeError: 'GeoAxes' object has no attribute '_autoscaleXon'

<Figure size 1100x520 with 1 Axes>

## 4 · XGBoost with leave-one-fold-out cross-validation
For each mode and each trait PC:

1. **Cross-validation.** Each fold is left out once. Inside the training data, a held-out 15% of folds is used for early stopping. The out-of-fold predictions give the honest R².
2. **Final model.** A model is fitted on all cells, with as many trees as the median early-stopping point. SHAP values come from this model, using exact TreeSHAP via `pred_contribs`, so the `shap` package isn't needed.

Runtime grows with folds × modes × PCs. On a laptop, expect roughly 5–20 minutes.

In [ ]:
def spatial_cv(X, y, folds):
    oof = np.full(len(y), np.nan)
    best_iters, fold_r2 = [], []
    inner = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    for k in np.unique(folds):
        test = folds == k
        tr = np.flatnonzero(~test)
        fit_i, stop_i = next(inner.split(tr, groups=folds[tr]))
        model = xgb.XGBRegressor(**XGB_PARAMS, early_stopping_rounds=EARLY_STOPPING)
        model.fit(X.iloc[tr[fit_i]], y[tr[fit_i]],
                  eval_set=[(X.iloc[tr[stop_i]], y[tr[stop_i]])], verbose=False)
        oof[test] = model.predict(X.iloc[test])
        best_iters.append(model.best_iteration + 1)
        if test.sum() > 1:
            fold_r2.append(r2_score(y[test], oof[test]))
    return oof, best_iters, fold_r2

def fit_final(X, y, n_trees):
    params = {**XGB_PARAMS, "n_estimators": int(n_trees)}
    model = xgb.XGBRegressor(**params).fit(X, y, verbose=False)
    contrib = model.get_booster().predict(xgb.DMatrix(X), pred_contribs=True)
    shap_df = pd.DataFrame(contrib[:, :-1], index=X.index, columns=X.columns)
    return model, shap_df, contrib[0, -1]

RESULTS = {}
for mode in MODES:
    for pc in TRAIT_PCS:
        y = TARGETS[mode].loc[CELLS, pc].to_numpy()
        oof, iters, fold_r2 = spatial_cv(FEATURES, y, FOLDS)
        n_trees = max(int(np.median(iters)), 10)
        model, shap_df, base = fit_final(FEATURES, y, n_trees)
        RESULTS[(mode, pc)] = dict(
            y=y, oof=oof, shap=shap_df, base=base, model=model, n_trees=n_trees,
            r2=r2_score(y, oof), rmse=np.sqrt(mean_squared_error(y, oof)),
            fold_r2=np.array(fold_r2),
            r2_train=r2_score(y, model.predict(FEATURES)),
        )
        r = RESULTS[(mode, pc)]
        print(f"{mode:8s} {pc}:  CV R² = {r['r2']:.3f}   (RMSE {r['rmse']:.3f}, "
              f"median fold R² {np.median(r['fold_r2']):.2f}, {n_trees} trees, train R² {r['r2_train']:.2f})")

In [ ]:
metrics = pd.DataFrame([
    dict(mode=m, trait_pc=pc, r2_cv=r["r2"], rmse_cv=r["rmse"],
         r2_fold_median=np.median(r["fold_r2"]), r2_fold_q25=np.quantile(r["fold_r2"], .25),
         r2_fold_q75=np.quantile(r["fold_r2"], .75), r2_train=r["r2_train"],
         n_trees=r["n_trees"], n_cells=len(r["y"]), n_folds=N_FOLDS)
    for (m, pc), r in RESULTS.items()])
metrics.to_csv(OUT_DIR / "cv_metrics.csv", index=False)
metrics.round(3)

## 5 · Results figures
### 5.1 How much of each trait axis do the predictors explain?
Bars show the pooled out-of-fold R². Whiskers show the interquartile range of the per-fold R². Low R² in *shift* mode is a result in its own right: it means the shifts are not predictable from climate, soil and land use alone.

The per-fold R² is often negative even when the pooled R² is good. Each fold covers only a narrow slice of trait space, so there is little variance within it to explain. Report the pooled value; the whiskers show how uneven skill is across regions.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
w = 0.8 / len(MODES)
x = np.arange(len(TRAIT_PCS))
for i, mode in enumerate(MODES):
    sub_m = metrics[metrics["mode"] == mode].set_index("trait_pc").loc[TRAIT_PCS]
    xs = x + (i - (len(MODES) - 1) / 2) * w
    ax.bar(xs, sub_m["r2_cv"], width=w * 0.9, color=MODE_COLORS[mode], label=mode, zorder=2)
    ax.errorbar(xs, sub_m["r2_fold_median"],
                yerr=[sub_m["r2_fold_median"] - sub_m["r2_fold_q25"], sub_m["r2_fold_q75"] - sub_m["r2_fold_median"]],
                fmt="o", ms=3, color=INK2, lw=0.8, zorder=3)
    for xi, v in zip(xs, sub_m["r2_cv"]):
        ax.text(xi, max(v, 0) + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=7, color=INK2)
ax.axhline(0, color=INK2, lw=0.8)
ax.set_xticks(x, TRAIT_PCS)
ax.set_ylabel("cross-validated R²")
ax.set_title("Predictive skill under spatial cross-validation", loc="left")
ax.legend(ncol=len(MODES), loc="upper right")
savefig(fig, "01_cv_r2")
plt.show()

### 5.2 Which predictors matter, and in which direction? (key figure)
**Left:** the share of total mean |SHAP| that each predictor contributes. Each column sums to 100%. This is the analogue of Mahecha et al. Fig. 4i–l.
**Right:** the direction of each effect, as the Spearman correlation between a predictor's value and its SHAP value. Blue means higher values of the predictor lower the trait PC; red means they raise it. Empty cells are predictors that carry less than 2% of the signal for that target, where a direction would be noise.

In [17]:
targets_order = [(m, pc) for m in MODES for pc in TRAIT_PCS]
col_labels = [f"{m}\n{pc}" for m, pc in targets_order]

importance = pd.DataFrame({f"{m}|{pc}": RESULTS[(m, pc)]["shap"].abs().mean() for m, pc in targets_order})
share = importance / importance.sum() * 100
direction = pd.DataFrame({
    f"{m}|{pc}": [spearmanr(FEATURES[c], RESULTS[(m, pc)]["shap"][c]).correlation for c in FEATURES.columns]
    for m, pc in targets_order}, index=FEATURES.columns)

row_order = share.mean(axis=1).sort_values(ascending=False).index
share, direction = share.loc[row_order], direction.loc[row_order]
MIN_SHARE = 2.0   # % — direction is only shown where a predictor carries at least this much signal
importance.loc[row_order].to_csv(OUT_DIR / "shap_mean_abs.csv")
share.to_csv(OUT_DIR / "shap_share_percent.csv")
direction.to_csv(OUT_DIR / "shap_direction_spearman.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 0.38 * len(row_order) + 2), gridspec_kw=dict(wspace=0.35))
for ax, data, cmap, norm, fmt, title, cbar_label in [
    (axes[0], share, SEQ, mpl.colors.Normalize(0, max(share.values.max(), 1)), "{:.0f}",
     "Share of mean |SHAP| (%)", "% of explained signal"),
    (axes[1], direction.where(share >= MIN_SHARE), DIV, TwoSlopeNorm(0, -1, 1), "{:+.1f}",
     "Direction of effect (ρ predictor vs SHAP)", "ρ"),
]:
    ax.grid(False)
    im = ax.imshow(data.values, cmap=cmap, norm=norm, aspect="auto")
    ax.set_xticks(range(len(col_labels)), col_labels, fontsize=7)
    ax.set_yticks(range(len(row_order)), [pretty(c) for c in row_order])
    for b in range(1, len(MODES)):
        ax.axvline(b * len(TRAIT_PCS) - 0.5, color=SURFACE, lw=3)
    for (i, j), v in np.ndenumerate(data.values):
        if np.isfinite(v):
            rgba = cmap(norm(v))
            lum = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=7,
                    color="white" if lum < 0.5 else INK)
    ax.set_title(title, loc="left")
    fig.colorbar(im, ax=ax, shrink=0.6, label=cbar_label)
savefig(fig, "02_shap_heatmaps")
plt.show()

NameError: name 'RESULTS' is not defined

### 5.3 SHAP summary per target
Each dot is one H3 cell. Its position shows how much a predictor pushes that cell's trait PC up or down. Its colour shows the predictor's value in that cell, from low (blue) to high (red), as percentile ranks.

In [ ]:
def beeswarm(ax, shap_df, X, max_display=10, title=""):
    order = shap_df.abs().mean().sort_values(ascending=False).index[:max_display][::-1]
    rng_b = np.random.default_rng(0)
    for row, feat in enumerate(order):
        s = shap_df[feat].to_numpy()
        v = X[feat].rank(pct=True).to_numpy()
        # jitter scaled by local density so dense regions spread out
        hist, edges = np.histogram(s, bins=60)
        dens = hist[np.clip(np.digitize(s, edges) - 1, 0, len(hist) - 1)] / hist.max()
        yj = row + rng_b.uniform(-0.38, 0.38, len(s)) * dens
        ax.scatter(s, yj, c=v, cmap=DIV, vmin=0, vmax=1, s=1.5, linewidths=0, rasterized=True)
    ax.axvline(0, color=INK2, lw=0.8)
    ax.set_yticks(range(len(order)), [pretty(f) for f in order])
    ax.set_xlabel("SHAP value (trait-PC units)")
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)

for mode in MODES:
    fig, axes = plt.subplots(1, len(TRAIT_PCS), figsize=(5.6 * len(TRAIT_PCS), 4.2), squeeze=False,
                             gridspec_kw=dict(wspace=0.55))
    for ax, pc in zip(axes[0], TRAIT_PCS):
        r = RESULTS[(mode, pc)]
        beeswarm(ax, r["shap"], FEATURES, title=f"{mode} · {pc}  (CV R² {r['r2']:.2f})")
    sm = mpl.cm.ScalarMappable(cmap=DIV, norm=mpl.colors.Normalize(0, 1))
    cb = fig.colorbar(sm, ax=axes[0], shrink=0.7, pad=0.01)
    cb.set_ticks([0, 1], labels=["low", "high"]); cb.set_label("predictor value (percentile)")
    savefig(fig, f"03_shap_beeswarm_{mode}")
    plt.show()

### 5.4 Dependence plots: response curves of the three strongest predictors
Each panel plots predictor value against SHAP value. The dark line is the binned median, which approximates the partial effect of that predictor.

In [ ]:
def binned_median(x, y, bins=20):
    q = np.unique(np.quantile(x, np.linspace(0, 1, bins + 1)))
    idx = np.clip(np.digitize(x, q) - 1, 0, len(q) - 2)
    xm = np.array([np.median(x[idx == b]) for b in range(len(q) - 1) if (idx == b).any()])
    ym = np.array([np.median(y[idx == b]) for b in range(len(q) - 1) if (idx == b).any()])
    return xm, ym

TOP_K = 3
for mode in MODES:
    fig, axes = plt.subplots(len(TRAIT_PCS), TOP_K, figsize=(4 * TOP_K, 3 * len(TRAIT_PCS)), squeeze=False)
    for i, pc in enumerate(TRAIT_PCS):
        sd = RESULTS[(mode, pc)]["shap"]
        for j, feat in enumerate(sd.abs().mean().sort_values(ascending=False).index[:TOP_K]):
            ax = axes[i, j]
            x, s = FEATURES[feat].to_numpy(), sd[feat].to_numpy()
            ax.scatter(x, s, s=2, color=MODE_COLORS[mode], alpha=0.25, linewidths=0, rasterized=True)
            ax.plot(*binned_median(x, s), color=INK, lw=2)
            ax.axhline(0, color=INK2, lw=0.6)
            ax.set_xlabel(pretty(feat)); ax.set_ylabel(f"SHAP → {pc}" if j == 0 else "")
            ax.set_title(f"{mode} · {pc}" if j == 0 else "", loc="left")
    fig.tight_layout()
    savefig(fig, f"04_shap_dependence_{mode}")
    plt.show()

### 5.5 Maps: observed, predicted (out-of-fold) and residuals
Look for spatial structure in the residuals. It shows where the predictors miss something, such as biogeographic history, sampling, or introduction pressure.

In [ ]:
def sym_limits(*arrays, q=0.98):
    v = np.nanquantile(np.abs(np.concatenate(arrays)), q)
    return -v, v

for mode in MODES:
    fig = plt.figure(figsize=(15, 2.7 * len(TRAIT_PCS)))
    fig.subplots_adjust(hspace=0.12, wspace=0.08)
    for i, pc in enumerate(TRAIT_PCS):
        r = RESULTS[(mode, pc)]
        resid = r["y"] - r["oof"]
        lo, hi = sym_limits(r["y"], r["oof"])
        rlo, rhi = sym_limits(resid)
        for j, (vals, title, lim) in enumerate([(r["y"], "observed", (lo, hi)),
                                               (r["oof"], "predicted (out-of-fold)", (lo, hi)),
                                               (resid, "residual", (rlo, rhi))]):
            ax, tf = map_axes(fig, (len(TRAIT_PCS), 3, i * 3 + j + 1))
            sc = ax.scatter(LON, LAT, c=vals, cmap=DIV, norm=TwoSlopeNorm(0, lim[0], lim[1]),
                            s=2, linewidths=0, rasterized=True, **tf)
            ax.set_title(f"{mode} · {pc} · {title}", loc="left", fontsize=9)
            fig.colorbar(sc, ax=ax, shrink=0.75, pad=0.01)
    savefig(fig, f"05_maps_{mode}")
    plt.show()

### 5.6 Residual diagnostics: does sampling effort affect accuracy?
These plots show the absolute out-of-fold residual against the number of records per cell and against population density. Lines are binned medians and bands are interquartile ranges. A falling curve means poorly sampled cells are predicted less well. This is a diagnostic only; nothing here feeds back into the model.

In [ ]:
diag_vars = {"log10 records per cell": np.log10(cell_info.loc[CELLS, "n_records_all"].to_numpy() + 1)}
if "log_pop_density" in FEATURES:
    diag_vars["log10 population density"] = FEATURES["log_pop_density"].to_numpy()

fig, axes = plt.subplots(len(diag_vars), len(TRAIT_PCS), figsize=(4.2 * len(TRAIT_PCS), 3.2 * len(diag_vars)),
                         squeeze=False, sharey="col")
for i, (xlabel, xv) in enumerate(diag_vars.items()):
    for j, pc in enumerate(TRAIT_PCS):
        ax = axes[i, j]
        for mode in MODES:
            ar = np.abs(RESULTS[(mode, pc)]["y"] - RESULTS[(mode, pc)]["oof"])
            q = np.unique(np.quantile(xv, np.linspace(0, 1, 11)))
            idx = np.clip(np.digitize(xv, q) - 1, 0, len(q) - 2)
            bins = [b for b in range(len(q) - 1) if (idx == b).any()]
            bx = [np.median(xv[idx == b]) for b in bins]
            bq = np.array([np.quantile(ar[idx == b], [.25, .5, .75]) for b in bins])
            rho = spearmanr(xv, ar).correlation
            ax.fill_between(bx, bq[:, 0], bq[:, 2], color=MODE_COLORS[mode], alpha=0.15, linewidth=0)
            ax.plot(bx, bq[:, 1], color=MODE_COLORS[mode], lw=2, label=f"{mode} (ρ {rho:+.2f})")
        ax.set_xlabel(xlabel); ax.set_ylabel("|residual|" if j == 0 else "")
        ax.set_title(pc if i == 0 else "", loc="left")
        ax.legend(fontsize=7)
fig.tight_layout()
savefig(fig, "06_residual_diagnostics")
plt.show()

## 6 · Save everything

In [ ]:
pred_long = pd.concat([
    pd.DataFrame({"h3_index": CELLS, "mode": m, "trait_pc": pc, "observed": r["y"], "predicted_oof": r["oof"],
                  "residual": r["y"] - r["oof"], "fold": FOLDS, "lat": LAT, "lon": LON})
    for (m, pc), r in RESULTS.items()])
shap_long = pd.concat([
    r["shap"].assign(mode=m, trait_pc=pc, base_value=r["base"]).reset_index()
    for (m, pc), r in RESULTS.items()])

def save_table(df, stem):
    try:
        df.to_parquet(OUT_DIR / f"{stem}.parquet", index=False); return f"{stem}.parquet"
    except ImportError:
        df.to_csv(OUT_DIR / f"{stem}.csv.gz", index=False); return f"{stem}.csv.gz"

saved = [save_table(pred_long, "predictions_oof"), save_table(shap_long, "shap_values"),
         save_table(FEATURES.reset_index(), "model_features")]
clim_load.to_csv(OUT_DIR / "climate_pca_loadings.csv")
soil_load.to_csv(OUT_DIR / "soil_pca_loadings.csv")
for (m, pc), r in RESULTS.items():
    r["model"].save_model(OUT_DIR / f"xgb_{m}_{pc}.json")

print("written to", OUT_DIR.resolve())
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(OUT_DIR))

## 7 · Methods paragraph (draft, filled from this run)

In [ ]:
lu_txt = f", and {LANDCOVER_SOURCE}" if USE_LANDCOVER else ""
print(f"""We related trait composition to the environment with gradient-boosted trees (XGBoost {xgb.__version__}).
Predictors were the first {len(clim_var)} principal components of {len(wc_cols)} standardised WorldClim 2.1 variables
({clim_var.sum():.0%} of variance), the first {len(soil_var)} principal components of {len(sg_cols)} standardised SoilGrids 2.0
variables ({soil_var.sum():.0%} of variance), log10-transformed population density ({POP_SOURCE}) as a proxy for
sampling intensity{lu_txt}. Models were fitted separately for each of {', '.join(TRAIT_PCS)} for present-day (all species),
native-only and shift (present − native) trait scores across {len(CELLS)} H3 resolution-3 cells.
Following Mahecha et al. (2021), spatial cross-validation folds were defined by affinity propagation on an
asymmetric similarity matrix whose lower triangle contained similarities in the space of the four leading trait PCs
and whose upper triangle contained geographic proximity, yielding {N_FOLDS} folds (after merging clusters
< {MIN_FOLD_SIZE} cells). Each fold was withheld once; the number of trees was set by early stopping on a
grouped inner validation split. Predictor contributions were quantified with exact TreeSHAP values from a model
refitted on all cells.""")